# INSTRUCTOR SOLUTIONS — DO NOT DISTRIBUTE

## DIY Task: Independent Regression Challenge

This shows one complete example (predicting temp_min from humidity and wind_speed) plus guidance for grading other student attempts.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

print("Libraries loaded!")

## Step 1: Load and Explore

In [ ]:
weather = pd.read_csv('medina_weather_2024.csv')
weather = weather.dropna()

print(f"Data shape: {weather.shape}")
print(f"\nColumns: {list(weather.columns)}")
print(f"\nBasic statistics:")
print(weather.describe().round(1))

## EXAMPLE SOLUTION: Predicting temp_min

Walkthrough predicted temp_max. We'll predict temp_min using humidity and wind_speed.

In [ ]:
# EXAMPLE: Predicting temp_min (not temp_max like the walkthrough)
target = 'temp_min'
featureList = ['humidity', 'wind_speed']

print(f"Target: {target}")
print(f"Features: {featureList}")
print(f"\nBasic stats for {target}:")
print(weather[target].describe().round(2))

In [ ]:
# Calculate correlations
print(f"\nHow features correlate with {target}:")
for feature in featureList:
    r = weather[feature].corr(weather[target])
    print(f"  {feature:15s} → r = {r:+.3f}")

print(f"\nInterpretation:")
print(f"  - Humidity: weak positive (more humid = slightly warmer lows)")
print(f"  - Wind speed: negative (wind can cool things down)")

## Build the Model

In [ ]:
X = weather[featureList]
y = weather[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Model trained!")
print(f"Training set: {len(X_train)} days")
print(f"Test set: {len(X_test)} days")

## Evaluate the Model

In [ ]:
predictions = model.predict(X_test)
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)

print(f"MODEL RESULTS:")
print(f"  R²:  {r2:.4f}  ({r2*100:.1f}% of {target} variation explained)")
print(f"  MAE: {mae:.3f} degrees F")

print(f"\nComparison to Walkthrough:")
print(f"  Walkthrough (temp_max from temp_min):  R² = 0.92")
print(f"  This model (temp_min from humidity/wind): R² = {r2:.2f}")
print(f"\nInterpretation: Lower R² because humidity and wind_speed are weaker")
print(f"predictors than temp_min. Multiple factors affect daily lows.")

## Visualize

In [ ]:
# Actual vs predicted
plt.figure(figsize=(8, 6))

minVal = y_test.min()
maxVal = y_test.max()
plt.plot([minVal, maxVal],
         [minVal, maxVal],
         'r--', linewidth=2, label='Perfect predictions')

plt.scatter(y_test, predictions,
            alpha=0.5, color='steelblue', s=40,
            label='Our predictions')

plt.xlabel(f'Actual {target} (°F)', fontsize=11)
plt.ylabel(f'Predicted {target} (°F)', fontsize=11)
plt.title(f'Predicting {target} from {featureList}', fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"More scatter than walkthrough (lower R²) as expected")

## Coefficients

In [ ]:
print(f"What the model learned:\n")
for i, feature in enumerate(featureList):
    print(f"{feature}:       {model.coef_[i]:+.4f}")
print(f"Intercept: {model.intercept_:.2f}")

print(f"\nInterpretation:")
print(f"  - Humidity ({model.coef_[0]:+.4f}): Small positive effect")
print(f"    Each 1% more humidity → {model.coef_[0]:.3f}° warmer low")
print(f"  - Wind speed ({model.coef_[1]:+.4f}): Negative effect")
print(f"    Each 1 mph more wind → {abs(model.coef_[1]):.3f}° cooler low"

## Student Analysis (Example)

In [ ]:
analysis = """
STUDENT RESPONSE EXAMPLE:

1. What did you predict?
   I predicted temp_min (daily low temperature) because the walkthrough 
   predicted temp_max, and I wanted to try something different. Lows are 
   important for predicting frost, heating costs, etc.

2. What features did you use?
   I chose humidity and wind_speed because they seemed likely to affect 
   how cold things get at night. More humid air holds heat, and wind 
   causes evaporative cooling.

3. How good is your model?
   R² = 0.28, which is MUCH LOWER than the walkthrough (R² = 0.92).
   This is expected because humidity and wind are weaker predictors 
   than temp_min is for temp_max. Daily lows depend on many factors: 
   cloud cover, season, location (valley vs hilltop), etc.

4. What did the model learn?
   Humidity has a tiny positive effect (more humid → slightly warmer lows)
   Wind has a negative effect (more wind → cooler lows).
   Wind_speed has bigger impact than humidity.

5. Comparison?
   My R² (0.28) is much lower than the walkthrough (0.92). This shows 
   that the walkthrough example was "cheating" — using same-day temperatures. 
   My model shows a more realistic prediction problem.

6. Real-world use?
   Maybe for general estimates, but not for precision. ±4 degrees error 
   (MAE) is too much for agricultural frost warnings. But it shows the 
   relationship exists.
"""
print(analysis)

## Other Valid Student Choices

Here are other legitimate target/feature combinations students might try:

In [ ]:
# Students could also try:
alternatives = [
    ('precipitation', ['temp_max', 'humidity'], 'Predicting rainfall from temperature/humidity'),
    ('humidity', ['temp_max', 'wind_speed'], 'Predicting humidity from temp/wind'),
    ('wind_speed', ['temp_max', 'precipitation'], 'Predicting wind from temp/rain'),
    ('temp_max', ['temp_min', 'wind_speed'], 'Predicting high from low/wind'),
]

print("OTHER VALID STUDENT ATTEMPTS:\n")
for target_alt, features_alt, description in alternatives:
    X_alt = weather[features_alt]
    y_alt = weather[target_alt]
    X_tr, X_te, y_tr, y_te = train_test_split(X_alt, y_alt, test_size=0.2, random_state=42)
    m_alt = LinearRegression()
    m_alt.fit(X_tr, y_tr)
    r2_alt = r2_score(y_te, m_alt.predict(X_te))
    
    print(f"{description}")
    print(f"  R² = {r2_alt:.3f}")
    print()

## Grading Rubric

**Model Building (25 points)**
- Chose a valid target (not temp_max): 5 pts
- Chose 2+ valid features: 5 pts
- Correct train/test split: 5 pts
- Model trains without errors: 5 pts
- R² and MAE calculated correctly: 5 pts

**Visualization (15 points)**
- Actual vs predicted scatter plot with diagonal line: 10 pts
- Properly labeled axes and title: 5 pts

**Analysis (35 points)**
- Clear answer: What target? Why? (5 pts)
- Clear answer: What features? Why? (5 pts)
- Correctly compared R² to walkthrough (5 pts)
- Correctly interpreted coefficients (5 pts)
- Thoughtful comparison and explanation (10 pts)

**Writing Quality (15 points)**
- Complete sentences: 5 pts
- Logical organization: 5 pts
- Shows understanding (not just copying): 5 pts

**Total: 90 points**

**Deductions:**
- Chose temp_max as target: -10 pts (violates requirement)
- Only 1 feature: -10 pts (needs 2+)
- Weak/missing analysis: -15 pts per missing section
- Unfinished notebook: -25 pts

**Notes for Grading:**
- Accept any reasonable target/feature combo
- R² can be low — that's OK if student understands why
- Most important: Understanding that lower R² doesn't mean failure
- Look for evidence student learned the concept, not just copied code